# Hybrid Retrieval RAG Experiments

Runs the Hybrid Retrieval RAG pipeline using Chroma and with k = 5
This has experiments 7 and 8 - Single Query Expansion and Multi Query Expansion

**Pipeline:** Query → Dense Retrieval (Cosine) + Sparse Retrieval (BM25) → Re-Ranking and select Top K → LLM Generation → Answer  
**Evaluation:** RAGAS and DeepEval metrics

In [5]:
import sys
sys.path.append("..")

import os
import time
import json
import pandas as pd
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import load_dataset
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
import config
from ast import literal_eval
from sentence_transformers import CrossEncoder
from deepeval.evaluate import DisplayConfig, AsyncConfig
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
from ragas import SingleTurnSample
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import (
    FaithfulnessMetric, ContextualRecallMetric,
    ContextualPrecisionMetric, AnswerRelevancyMetric, GEval
)
import deepeval
import instructor
from groq import AsyncGroq, Groq

from ragas.llms.base import InstructorLLM
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()
pd.set_option('display.html.use_mathjax', False)

import logging
logging.basicConfig(level=logging.ERROR)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_2575/3296000977.py:22: DeprecationWarning: Importing NonLLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextRecall
  from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
/tmp/ipykernel_2575/3296000977.py:22: DeprecationWarning: Importing NonLLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.

## Load Vector Stores

Loaders for each vector DB ingested by `ingestion_pipeline.ipynb`. All functions are
read-only — they never re-embed or re-write.

In [ ]:
def load_chroma(embeddings, db_name=None, persist_dir=None):
    """Load an existing ChromaDB collection from disk.

    Args:
        embeddings: LangChain embeddings instance (must match what was used during ingestion).
        db_name: Collection name; defaults to {DEFAULT_EMBEDDING}_pubmed_chroma.
        persist_dir: Override storage path (defaults to vectorstores/{db_name}).

    Returns:
        Chroma vector store instance.
    """
    from langchain_chroma import Chroma

    db_name = db_name or f"{config.DEFAULT_EMBEDDING}_pubmed_chromadb"
    persist_dir = persist_dir or str(config.VECTORSTORE_DIR / db_name)
    print(f"Loading ChromaDB from {persist_dir}")
    return Chroma(
        collection_name=db_name,
        persist_directory=persist_dir,
        embedding_function=embeddings,
    )

## Create Hybrid Retriever

In [ ]:
def get_bm25_retriever(vector_store, k=None):
    """Method to build a BM25 lexical similarity based retriever for given vector store

    Loads all stored chunks from the vector store into memory as LangChain
    Document objects, then constructs a BM25 retriever for keyword-based
    sparse retrieval.

    Args:
        vector_store: LangChain vector store object which supports get()
            for retrieving stored documents and metadata.
        k: Top k items to be retrieved.

    Returns:
        BM25 retriever object.
    """
    k = k or config.TOP_K
    docs = [Document(page_content=x, metadata=m) for x, m in zip(vector_store.get()["documents"], vector_store.get()["metadatas"])]
    retriever = BM25Retriever.from_documents(docs)
    retriever.k = k
    return retriever

def get_cosine_retriever(vector_store, k=None):
    """Method to build a cosine similarity based retriever for given vector store
    Args:
        vector_store: Langchain vectore store object which has method as_retriever
        k: Top k items to be retrieved
    Returns:
        retriever object
    """
    k = k or config.TOP_K
    return vector_store.as_retriever(search_kwargs={"k": k})


def rrf(rank_lists, top_k=3, k=60):
    """Method to fuse multiple ranked retrieval lists using Reciprocal Rank Fusion.

    Applies Reciprocal Rank Fusion (RRF) to combine retrieval results from
    multiple retrievers while preserving rank information. Duplicate chunks
    are identified using (pubid, chunk_index) metadata and their RRF scores
    are accumulated across retrieval lists.

    Args:
        rank_lists: List of ranked document lists returned by retrievers.
        top_k: Final number of top ranked chunks to return after fusion.
        k: RRF smoothing constant. Higher values reduce the effect of rank.

    Returns:
        List of fused and reranked LangChain Document objects.
    """
    scores = defaultdict(lambda: {"doc": None, "score": 0.0})
    for docs in rank_lists:
        for rank, doc in enumerate(docs, 1):
            key = (doc.metadata["pubid"], doc.metadata["chunk_index"])
            # Only store the document if it's the first time we see it (highest rank)
            # This preserves the metadata associated with its best retrieval rank
            if scores[key]["doc"] is None:
                scores[key]["doc"] = doc
            scores[key]["score"] += 1 / (k + rank)
    return [x["doc"] for x in sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_k]]


def invoke_hybrid_retriever(query, dense_retriever, sparse_retriever, top_k=None):
    """Method to perform hybrid retrieval using dense and sparse retrievers.

    Retrieves candidate chunks independently using dense semantic retrieval
    (cosine similarity) and sparse lexical retrieval (BM25), then combines
    both ranked lists using Reciprocal Rank Fusion (RRF).

    Args:
        query: User query string to retrieve relevant chunks for.
        dense_retriever: Dense retriever object based on vector similarity.
        sparse_retriever: Sparse retriever object based on BM25 scoring.
        top_k: Final number of top ranked chunks to return after fusion.

    Returns:
        List of fused and reranked LangChain Document objects.
    """
    top_k = top_k or config.TOP_K
    dense_docs = dense_retriever.invoke(query)
    sparse_docs = sparse_retriever.invoke(query)
    return rrf([dense_docs, sparse_docs], top_k=top_k)

## Hybrid RAG Chain with RRF

In [6]:
def get_groq_llm(model=None, api_key=None):
    return ChatGroq(
        model=model or config.LLM_MODEL,
        api_key=api_key or config.GROQ_API_KEY,
    )


class GroqKeyRotator:
    """Cycles through config.GROQ_API_KEYS, rebuilding the LLM client on each rotation."""

    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty — set GROQ_API_KEY or GROQ_API_KEYS in .env")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        self.current_idx = 0
        print(f"Initialized GroqKeyRotator with {len(self.api_keys)} API key(s)")

    def get_llm(self):
        """Return a ChatGroq instance using the current API key."""
        return ChatGroq(
            model=self.model,
            api_key=self.api_keys[self.current_idx],
        )

    def rotate(self):
        """Advance to the next key, wrapping around."""
        self.current_idx = (self.current_idx + 1) % len(self.api_keys)
        print(f"Rotated to API key index {self.current_idx}")


def is_rate_limit_error(e):
    """Check if rate limit error is reached by checking the error message."""
    msg = str(e).lower()
    return any(kw in msg for kw in [
        "rate_limit", "rate limit", "429", "too many requests",
        "tokens per", "token limit", "exceeded",
    ])


RAG_PROMPT_TEMPLATE = """Use the following research contexts to answer the question.

Context:
{context}

Question: {question}

Answer based only on the provided context. Be precise and evidence-based.

Answer:"""

RAG_PROMPT = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)


def build_naive_rag_chain(llm):
    return RAG_PROMPT | llm


def _run_slice(retrievers, slice_df, api_key, model, delay, key_idx):
    """Process a contiguous slice of the evaluation set using a single dedicated API key.

    Called in its own thread by run_rag_parallel. Rows are processed sequentially
    with `delay` seconds between requests to stay within the key's daily quota.
    Adds retrieved_contexts and generated_answer as new columns to a copy of slice_df.

    Args:
        retrievers: Tuple of dense and sparse retrievers
        slice_df: Contiguous DataFrame slice assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name forwarded from GroqKeyRotator.
        delay: Seconds to sleep between rows.
        key_idx: Key index used only for log prefixes.

    Returns:
        Copy of slice_df with two new columns: retrieved_contexts and generated_answer.
    """
    chain = build_naive_rag_chain(ChatGroq(model=model, api_key=api_key))
    # Get Dense and Sparse retrievers from tuple
    dense_retriever, sparse_retriever = retrievers

    result_df = slice_df.copy().reset_index(drop=True)
    # Creating empty lists
    retrieved_contexts_list = [None] * len(slice_df)
    generated_answer_list = [None] * len(slice_df)
    total_time_list = [None] * len(slice_df)
    prompt_tokens_list = [None] * len(slice_df)
    completion_tokens_list = [None] * len(slice_df)
    total_tokens_list = [None] * len(slice_df)

    for row_idx, (_, row) in enumerate(slice_df.iterrows()):
        question = row["question"]
        try:
            time_start = time.perf_counter()
            # Retrieval
            contexts = invoke_hybrid_retriever(question, dense_retriever, sparse_retriever)

            # Generation
            result = chain.invoke({"context": contexts, "question": question})
            total_time = time.perf_counter() - time_start

            # Token usage
            token_usage = result.response_metadata.get("token_usage", {})
            prompt_tokens = token_usage.get("prompt_tokens", 0)
            completion_tokens = token_usage.get("completion_tokens", 0)
            total_tokens = token_usage.get("total_tokens", 0)

            retrieved_contexts_list[row_idx] = [doc.page_content for doc in contexts]
            generated_answer_list[row_idx] = result.content
            total_time_list[row_idx] = total_time
            prompt_tokens_list[row_idx] = prompt_tokens
            completion_tokens_list[row_idx] = completion_tokens
            total_tokens_list[row_idx] = total_tokens
        except Exception as e:
            print(f"[Key {key_idx}] Error on '{question[:50]}...': {e}")

        if row_idx < len(slice_df) - 1:
            time.sleep(delay)

    result_df["retrieved_contexts"] = retrieved_contexts_list
    result_df["generated_answer"] = generated_answer_list
    result_df["total_time"] = total_time_list
    result_df["prompt_tokens"] = prompt_tokens_list
    result_df["completion_tokens"] = completion_tokens_list
    result_df["total_tokens"] = total_tokens_list

    completed = sum(1 for x in generated_answer_list if x is not None)
    print(f"[Key {key_idx}] Done — {completed}/{len(slice_df)} rows collected")
    return result_df


def run_rag_parallel(retrievers, df, key_rotator, rows_per_key=None, delay=None):
    """Assign a contiguous slice of rows to each API key and run all slices in parallel.

    Key 0 gets rows 0..rows_per_key-1, key 1 gets the next rows_per_key rows, etc.
    Each key runs in its own thread; since Groq keys have independent daily quotas,
    the threads don't interfere with each other.

    ThreadPoolExecutor is used (not asyncio) because: (1) network I/O releases the GIL
    so threads genuinely run concurrently, (2) Jupyter/Colab already have a running event
    loop so asyncio.run() raises RuntimeError, and (3) ChatGroq.invoke() is synchronous.

    Args:
        retrievers: Tuple of dense and sparse retrievers
        df: DataFrame with at least question and golden_answer columns.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        rows_per_key: Max rows assigned to each key (default config.PARALLEL_BUCKET_SIZE).
        delay: Seconds between rows within each key's slice (default config.PARALLEL_DELAY_SECONDS).

    Returns:
        a copy of df with new columns
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.PARALLEL_DELAY_SECONDS
    api_keys = key_rotator.api_keys

    if len(df) == 0:
        raise ValueError("DataFrame is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(df) > total_capacity:
        print(
            f"Warning: {len(df)} rows exceed capacity ({len(api_keys)} keys × {rows_per_key} rows = "
            f"{total_capacity}). Only the first {total_capacity} rows will be processed."
        )
        df = df.iloc[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(df):
            break
        slices.append((key, i, df.iloc[start: start + rows_per_key]))

    print(f"\n{len(df)} rows split across {len(slices)} key(s) ({rows_per_key} rows/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: rows {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} rows)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_slice,
                retrievers, s, key, key_rotator.model, delay, i,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            _, _, s = slices[idx]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                fallback = s.copy().reset_index(drop=True)
                fallback["retrieved_contexts"] = [None] * len(s)
                fallback["generated_answer"] = [None] * len(s)
                ordered_results[idx] = fallback

    final_df = pd.concat(ordered_results, ignore_index=True)
    completed = final_df["generated_answer"].notna().sum()
    print(f"\nCompleted {completed}/{len(df)} questions total")
    print(f"\nAverage Time Per Query : {sum(final_df["total_time"])/len(final_df)}")
    print(f"\nAverage Total Tokens Per Query : {sum(final_df["total_tokens"])/len(final_df)}")

    return final_df

## Evaluation Functions

In [7]:
class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model=None):
        self.model = model or get_groq_llm()

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "Groq Model"


def build_test_cases(eval_df):
    return [
        LLMTestCase(
            input=row["question"],
            actual_output=row["generated_answer"],
            retrieval_context=row["retrieved_contexts"],
            expected_output=row["golden_answer"],
        )
        for _, row in eval_df.iterrows()
    ]


def _make_ragas_scores_df(all_scores, metric_name):
    """Return a minimal DataFrame with question_index and metric score."""
    return pd.DataFrame({
        "question_index": range(len(all_scores)),
        metric_name: all_scores,
    })


def evaluate_ragas(eval_df, metric, results_file=None):
    """Evaluate with a non-LLM RAGAS metric over each row of the eval DataFrame.

    Uses SingleTurnSample + single_turn_score (synchronous) — no API keys required.
    retrieved_contexts (what RAG retrieved) is compared against reference_contexts
    (the PubMedQA golden reference contexts).

    Args:
        eval_df: DataFrame produced by run_rag / run_rag_parallel.
            Expected columns: question, generated_answer, retrieved_contexts,
            golden_contexts, golden_answer.
        metric: A RAGAS metric instance.
        results_file: Optional CSV path to save per-sample scores (question_index + score only).
        is_old_metric_type: Boolean to indicate whether the metric is from old
        collections package

    Returns:
        Tuple of (list of per-sample scores, average score, scores DataFrame).
    """
    metric_name = type(metric).__name__
    all_scores = []

    for _, row in eval_df.iterrows():
        sample = SingleTurnSample(
            user_input=row["question"],
            retrieved_contexts=list(row["retrieved_contexts"]),
            reference_contexts=row["golden_contexts"],
            reference=row["golden_answer"],
            response=row["generated_answer"]
        )
        score = metric.single_turn_score(sample)
        all_scores.append(score)

    avg = sum(all_scores) / len(all_scores)
    print(f"\n=== {metric_name}: {avg:.4f} (avg over {len(all_scores)} samples) ===")

    scores_df = _make_ragas_scores_df(all_scores, metric_name)

    if results_file:
        scores_df.to_csv(results_file, index=False)
        print(f"Saved scores to {results_file}")

    return all_scores, avg, scores_df


def _run_ragas_llm_slice(eval_df_slice, api_key, model, metric_cls, embeddings, delay, key_idx):
    """Evaluate a contiguous slice of rows with an LLM-based RAGAS metric.

    Called in its own thread by evaluate_ragas_parallel. Each thread creates its own
    IntructorLLM + metric instance and its own asyncio event loop, so threads
    never share state and asyncio.run() never conflicts with Jupyter's main loop.

    Args:
        eval_df_slice: Contiguous DataFrame slice assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name forwarded from GroqKeyRotator.
        metric_cls: LLM-based RAGAS metric class (not an instance), e.g. ContextRecall.
        delay: Seconds to sleep between rows.
        key_idx: Key index used only for log prefixes.

    Returns:
        List of per-row scores (float or None on error) in slice order.
    """
    client = instructor.from_groq(
        AsyncGroq(api_key=api_key),
        mode=instructor.Mode.JSON,
    )

    ragas_llm = InstructorLLM(
        client=client,
        provider='groq',
        model=model,
        is_async=True,
    )
    metric = metric_cls(llm=llm, embeddings=embeddings)
    scores = []

    for row_idx, (_, row) in enumerate(eval_df_slice.iterrows()):
        try:
            sample = SingleTurnSample(
                user_input=row["question"],
                retrieved_contexts=row["retrieved_contexts"],
                reference_contexts=row["golden_contexts"],
                reference=row["golden_answer"],
                response=row["generated_answer"],
            )
            score = metric.single_turn_score(sample)
            scores.append(score)
        except Exception as e:
            print(f"[Key {key_idx}] Error on row {row_idx}: {e}")
            scores.append(None)

        if row_idx < len(eval_df_slice) - 1:
            time.sleep(delay)

    completed = sum(1 for s in scores if s is not None)
    print(f"[Key {key_idx}] Done — {completed}/{len(eval_df_slice)} rows scored")
    return scores


def evaluate_ragas_parallel(eval_df, metric_cls, key_rotator, embeddings, results_file=None, delay=None, rows_per_key=None):
    """Evaluate an LLM-based RAGAS metric in parallel, one API key per slice.

    Key 0 gets rows 0..rows_per_key-1, key 1 gets the next slice, etc.
    Returns the same (scores, avg, scores_df) tuple as evaluate_ragas, so the
    result plugs directly into build_ragas_combined as a drop-in replacement.

    Args:
        eval_df: DataFrame produced by run_rag / run_rag_parallel.
        metric_cls: LLM-based RAGAS metric class (not an instance), e.g. ContextRecall.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        results_file: Optional CSV path to save per-sample scores (question_index + score only).
        delay: Seconds between rows within each slice (default config.PARALLEL_DELAY_SECONDS).
        rows_per_key: Max rows assigned to each key (default config.PARALLEL_BUCKET_SIZE).

    Returns:
        Tuple of (list of per-sample scores, average score, scores DataFrame).
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.PARALLEL_DELAY_SECONDS
    api_keys = key_rotator.api_keys
    metric_name = metric_cls.__name__

    if len(eval_df) == 0:
        raise ValueError("eval_df is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(eval_df) > total_capacity:
        print(
            f"Warning: {len(eval_df)} rows exceed capacity ({len(api_keys)} keys × {rows_per_key} = "
            f"{total_capacity}). Only the first {total_capacity} rows will be processed."
        )
        eval_df = eval_df.iloc[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(eval_df):
            break
        slices.append((key, i, eval_df.iloc[start: start + rows_per_key]))

    print(f"\n{len(eval_df)} rows split across {len(slices)} key(s) ({rows_per_key} rows/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: rows {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} rows)")
    print()

    ordered_scores = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_ragas_llm_slice,
                s, key, key_rotator.model, metric_cls, embeddings, delay, i,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            _, _, s = slices[idx]
            try:
                ordered_scores[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                ordered_scores[idx] = [None] * len(s)

    all_scores = []
    for slice_scores in ordered_scores:
        all_scores.extend(slice_scores)

    valid = [s for s in all_scores if s is not None]
    avg = sum(valid) / len(valid) if valid else 0.0
    print(f"\n=== {metric_name}: {avg:.4f} (avg over {len(valid)} samples) ===")

    scores_df = _make_ragas_scores_df(all_scores, metric_name)

    if results_file:
        scores_df.to_csv(results_file, index=False)
        print(f"Saved scores to {results_file}")

    return all_scores, avg, scores_df

def build_ragas_combined(eval_df, score_dfs, results_file=None):
    """Combine eval_df with per-metric score DataFrames into one summary CSV.

    Args:
        eval_df: DataFrame produced by run_rag / run_rag_parallel.
        score_dfs: List of score DataFrames from evaluate_ragas, each with
            question_index + one metric score column.
        results_file: Optional CSV path to save the combined DataFrame.

    Returns:
        Combined DataFrame with question_index, question, retrieved_contexts,
        golden_contexts, golden_answer, generated_response, and one column per metric.
    """
    combined = eval_df.copy().reset_index(drop=True)
    combined.insert(0, "question_index", range(len(combined)))
    combined = combined.rename(columns={"generated_answer": "generated_response"})

    for scores_df in score_dfs:
        combined = combined.merge(scores_df, on="question_index", how="left")

    if results_file:
        combined.to_csv(results_file, index=False)
        print(f"Saved combined RAGAS results to {results_file}")

    return combined


def _run_deepeval_slice(test_case_slice, api_key, model, metric_cls, threshold, delay, key_idx, metric_kwargs=None):
    """Evaluate a contiguous slice of test cases using a single dedicated API key.

    Called in its own thread by evaluate_deepeval_parallel. Test cases are evaluated
    sequentially with `delay` seconds between each to stay within the key's daily quota.

    Args:
        test_case_slice: List of LLMTestCase objects assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name forwarded from GroqKeyRotator.
        metric_cls: DeepEval metric class (e.g. ContextualRecallMetric), not an instance.
        threshold: Pass/fail threshold for the metric.
        delay: Seconds to sleep between test cases.
        key_idx: Key index used only for log prefixes.

    Returns:
        List of deepeval test results in slice order.
    """
    llm = ChatGroq(model=model, api_key=api_key)
    results = []

    for i, test_case in enumerate(test_case_slice):
        try:
            metric = metric_cls(threshold=threshold, model=GroqModel(model=llm),
                                **metric_kwargs)
            result = deepeval.evaluate([test_case], metrics=[metric],
                                       display_config= DisplayConfig(
                                           verbose_mode=False,
                                           show_indicator=False,
                                           print_results=False),
                                       async_config=AsyncConfig(run_async=False))
            results.extend(result.test_results)
        except Exception as e:
            print(f"[Key {key_idx}] Error on case {i + 1}/{len(test_case_slice)}: {e}")

        if i < len(test_case_slice) - 1:
            time.sleep(delay)

    print(f"[Key {key_idx}] Done — {len(results)}/{len(test_case_slice)} cases evaluated")
    return results


def evaluate_deepeval_parallel(test_cases, metric_cls, key_rotator, threshold=0.5, results_file=None, delay=None, rows_per_key=None, metric_kwargs=None):
    """Assign a contiguous slice of test cases to each API key and run all slices in parallel.

    Key 0 gets test_cases[0:rows_per_key], key 1 gets the next slice, etc.
    Each key runs in its own thread; since Groq keys have independent daily quotas,
    the threads don't interfere with each other.

    Args:
        test_cases: List of LLMTestCase objects built by build_test_cases.
        metric_cls: DeepEval metric class (e.g. ContextualRecallMetric), not an instance.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        threshold: Pass/fail threshold for the metric (default 0.5).
        results_file: Optional CSV path to save per-sample scores.
        delay: Seconds between cases within each slice (defaults to config.DEEPEVAL_DELAY_SECONDS).
        rows_per_key: Max cases assigned to each key (default config.PARALLEL_BUCKET_SIZE).

    Returns:
        List of deepeval test results in original case order.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.DEEPEVAL_DELAY_SECONDS
    api_keys = key_rotator.api_keys
    metric_kwargs = {} if metric_kwargs is None else dict(metric_kwargs)

    if not test_cases:
        raise ValueError("test_cases is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(test_cases) > total_capacity:
        print(
            f"Warning: {len(test_cases)} cases exceed capacity ({len(api_keys)} keys × {rows_per_key} = "
            f"{total_capacity}). Only the first {total_capacity} cases will be processed."
        )
        test_cases = test_cases[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(test_cases):
            break
        slices.append((key, i, test_cases[start: start + rows_per_key]))

    print(f"\n{len(test_cases)} cases split across {len(slices)} key(s) ({rows_per_key} cases/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: cases {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} cases)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_deepeval_slice,
                s, key, key_rotator.model,
                metric_cls, threshold, delay, i, metric_kwargs
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                ordered_results[idx] = []

    all_results = []
    res_df = None
    for key_results in ordered_results:
        all_results.extend(key_results)

    if all_results:
        scores = [r.metrics_data[0].score for r in all_results]
        metric_name = all_results[0].metrics_data[0].name
        average = sum(scores) / len(scores)
        print(f"\n=== {metric_name}: {average:.4f} (avg over {len(scores)} samples) ===")

        rows = []
        for r in all_results:
            rows.append({
                "question": r.input,
                "generated_answer": r.actual_output,
                "retrieved_contexts": r.retrieval_context,
                "golden_answer": r.expected_output,
                r.metrics_data[0].name: r.metrics_data[0].score,
            })
        res_df = pd.DataFrame(rows)
        if results_file:
            res_df.to_csv(results_file, index=False)
            print(f"Saved to {results_file}")

    return all_results, res_df

---
## Setup

In [ ]:
embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
key_rotator = GroqKeyRotator()
llm = key_rotator.get_llm()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Embedding: {config.EMBEDDING_MODELS[embedding_key]}")
print(f"LLM: {key_rotator.model}")
print(f"Run timestamp: {timestamp}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized GroqKeyRotator with 10 API key(s)
Embedding: sentence-transformers/all-MiniLM-L6-v2
LLM: llama-3.3-70b-versatile
Run timestamp: 20260510_081037


## Prepare Evaluation Sample

Reads the pre-built golden dataset from `data/processed/golden_dataset_complete.csv`.
Generate this file once by running `python3 sampling.py` from the `research/` directory.
Keeping the split fixed is critical — regenerating mid-experiment would change which
questions each RAG variant sees, invalidating cross-experiment comparisons.

In [ ]:
golden_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_complete.csv")
print(f"Loaded {len(golden_df)} samples from golden_dataset_complete.csv")
print(f"Structure of golden dataset")
golden_df['golden_contexts'] = golden_df['golden_contexts'].apply(literal_eval)
print(type(golden_df['golden_contexts'].iloc[0]))
golden_df.info()

Loaded 200 samples from golden_dataset_complete.csv
Structure of golden dataset
<class 'list'>
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   question_idx     200 non-null    int64 
 1   question         200 non-null    object
 2   golden_answer    200 non-null    object
 3   golden_contexts  200 non-null    object
 4   query_type       200 non-null    object
 5   pubids_needed    200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB


## Load Vector Store

Load the pre-ingested vector store. Swap `load_chroma` for `load_faiss`, `load_qdrant`,
or `load_lancedb` to evaluate a different backend — everything downstream stays the same.

In [ ]:
# Using chromaDB embeddings to build a vector store
vector_store = load_chroma(embeddings, db_name=f"{embedding_key}_pubmed_chromadb")

Loading ChromaDB from /content/vectorstores/minilm_pubmed_chromadb


### Build Retrievers

In [ ]:
cosine_retriever = get_cosine_retriever(vector_store, k=5)
bm25_retriever = get_bm25_retriever(vector_store, k=5)

## Run Single Query Expansion RAG

In [ ]:
eval_dataset = run_rag_parallel((cosine_retriever, bm25_retriever), golden_df, key_rotator)
eval_dataset.to_csv(str(config.RESULTS_EVALSETS_DIR / f"hybrid_rrf_rag_{embedding_key}_chroma_cosine_{timestamp}.csv"))
print(f"Generated {len(eval_dataset)} answers")


200 rows split across 10 key(s) (20 rows/key max):
  Key 0: rows 0–19 (20 rows)
  Key 1: rows 20–39 (20 rows)
  Key 2: rows 40–59 (20 rows)
  Key 3: rows 60–79 (20 rows)
  Key 4: rows 80–99 (20 rows)
  Key 5: rows 100–119 (20 rows)
  Key 6: rows 120–139 (20 rows)
  Key 7: rows 140–159 (20 rows)
  Key 8: rows 160–179 (20 rows)
  Key 9: rows 180–199 (20 rows)

[Key 2] Done — 20/20 rows collected
[Key 1] Done — 20/20 rows collected
[Key 4] Done — 20/20 rows collected
[Key 3] Done — 20/20 rows collected
[Key 6] Done — 20/20 rows collected
[Key 5] Done — 20/20 rows collected
[Key 0] Done — 20/20 rows collected
[Key 9] Done — 20/20 rows collected
[Key 7] Done — 20/20 rows collected
[Key 8] Done — 20/20 rows collected

Completed 200/200 questions total

Average Time Per Query : 4.240256823935002

Average Total Tokens Per Query : 1804.78
Generated 200 answers


### RAGAS Evaluation

In [ ]:
ragas_cr_scores, ragas_cr_avg, ragas_cr_df = evaluate_ragas(
    eval_dataset, NonLLMContextRecall(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"hybrid_rrf_rag_{embedding_key}_context_recall_{timestamp}.csv")
)


=== NonLLMContextRecall: 0.1967 (avg over 200 samples) ===
Saved scores to /content/results/ragas/hybrid_rrf_rag_minilm_context_recall_20260510_081037.csv


In [ ]:
ragas_cp_scores, ragas_cp_avg, ragas_cp_df = evaluate_ragas(
    eval_dataset, NonLLMContextPrecisionWithReference(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"hybrid_rrf_rag_{embedding_key}_context_precision_{timestamp}.csv")
)


=== NonLLMContextPrecisionWithReference: 0.2861 (avg over 200 samples) ===
Saved scores to /content/results/ragas/hybrid_rrf_rag_minilm_context_precision_20260510_081037.csv


In [ ]:
ragas_blue_scores, ragas_blue_avg, ragas_blue_df = evaluate_ragas(
    eval_dataset, BleuScore(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"hybrid_rrf_rag_{embedding_key}_bleu_{timestamp}.csv")
)


=== BleuScore: 0.1698 (avg over 200 samples) ===
Saved scores to /content/results/ragas/hybrid_rrf_rag_minilm_bleu_20260510_081037.csv


In [ ]:
ragas_rouge_scores, ragas_rouge_avg, ragas_rouge_df = evaluate_ragas(
    eval_dataset, RougeScore(rouge_type="rougeL", mode="fmeasure"),
    results_file=str(config.RESULTS_RAGAS_DIR / f"hybrid_rrf_rag_{embedding_key}_rouge_{timestamp}.csv")
)


=== RougeScore: 0.2970 (avg over 200 samples) ===
Saved scores to /content/results/ragas/hybrid_rrf_rag_minilm_rouge_20260510_081037.csv


In [ ]:
combined_ragas = build_ragas_combined(
    eval_dataset, [ragas_cr_df, ragas_cp_df, ragas_blue_df, ragas_rouge_df], results_file=str(config.RESULTS_RAGAS_DIR / f"hybrid_rrf_rag_{embedding_key}_combined_{timestamp}.csv"))

Saved combined RAGAS results to /content/results/ragas/hybrid_rrf_rag_minilm_combined_20260510_081037.csv


### DeepEval Evaluation

In [16]:
timestamp = "20260510_081037"
embedding_key = config.DEFAULT_EMBEDDING
eval_dataset = pd.read_csv(str(config.RESULTS_EVALSETS_DIR / f"hybrid_rrf_rag_{embedding_key}_chroma_cosine_{timestamp}.csv"), index_col=0)
eval_dataset['golden_contexts'] = eval_dataset['golden_contexts'].apply(literal_eval)
eval_dataset['retrieved_contexts'] = eval_dataset['retrieved_contexts'].apply(literal_eval)
eval_dataset.head(2)

,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,retrieved_contexts,generated_answer,total_time,prompt_tokens,completion_tokens,total_tokens
0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841'],[CONCLUSIONS: Based on data derived from self-...,"Yes, there is evidence to suggest a relationsh...",0.988384,1650,132,1782
1,1,"Do the changes in the serum levels of IL-2, IL...",The enhancement of serum TNFalpha and IL-6 lev...,[Acute pancreatitis is the major complication ...,Single-hop,['18670651'],[RESULTS: Seven of the 45 patients (15.5%) dev...,"Yes, the changes in the serum levels of IL-2, ...",0.857364,1811,173,1984


In [17]:
test_cases = build_test_cases(eval_dataset)

de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")

Initialized GroqKeyRotator with 10 API key(s)


In [ ]:
deepeval_cr, deepeval_cr_df = evaluate_deepeval_parallel(
    test_cases, ContextualRecallMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_rrf_rag_{embedding_key}_ctx_recall_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



[Key 8] Done — 20/20 cases evaluated

=== Contextual Recall: 0.9183 (avg over 200 samples) ===
Saved to /content/results/deepeval/hybrid_rrf_rag_minilm_ctx_recall_20260510_081037.csv


In [ ]:
deepeval_cp, deepeval_cp_df = evaluate_deepeval_parallel(
    test_cases, ContextualPrecisionMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_rrf_rag_{embedding_key}_ctx_precision_{timestamp}.csv")
)

In [ ]:
deepeval_f, deepeval_f_df = evaluate_deepeval_parallel(
    test_cases, FaithfulnessMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_rrf_rag_{embedding_key}_faithfulness_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



⚠ WARNING: No hyperparameters logged.
» ]8;id=717475;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 6.07s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

⚠ WARNING: No hyperparameters logged.
» ]8;id=868026;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 28.66s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

⚠ WARNING: No hyperparameters logged.
» ]8;id=224580;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 6.55s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 8] Done — 20/20 cases evaluated

=== Faithfulness: 0.9821 (avg over 194 samples) ===
Saved to /content/results/deepeval/hybrid_rrf_rag_minilm_faithfulness_20260510_081037.csv


In [10]:
def resume_deepeval_from_csv(eval_dataset, existing_results_file, metric_cls, key_rotator,
    metric_column, threshold=0.5, delay=None, rows_per_key=None, metric_kwargs=None):
    """Resume an interrupted DeepEval run from an existing CSV.

    Identifies missing or incomplete questions in an existing DeepEval results
    file, recomputes only those rows, merges the new scores, and overwrites the original file.

    Matching is performed using the question text rather than row position,
    making the method robust to out-of-order, partially completed, or shuffled CSV files.

    Args:
        eval_dataset: Full evaluation DataFrame.
        existing_results_file: Existing DeepEval CSV path.
        metric_cls: DeepEval metric class.
        key_rotator: API key rotator.
        metric_column: Metric column name in CSV.
        threshold: DeepEval threshold.
        delay: Delay between API calls.
        rows_per_key: Rows per API key.
        metric_kwargs: Optional metric init kwargs.

    Returns:
        Final merged DataFrame.
    """
    existing_df = pd.read_csv(existing_results_file)

    # Questions already successfully evaluated
    completed_questions = set(existing_df.loc[
        existing_df[metric_column].notna(),"question"].astype(str))

    # Missing/incomplete rows anywhere in dataset
    missing_df = eval_dataset[~eval_dataset["question"].astype(str)
                              .isin(completed_questions)].copy()
    if missing_df.empty:
        print("All rows already completed.")
        return existing_df
    print(f"Need to recompute {len(missing_df)} rows.")

    # Build test cases only for missing rows
    test_cases = build_test_cases(missing_df)

    # Run DeepEval only for missing rows
    _, new_results_df = evaluate_deepeval_parallel(test_cases, metric_cls,
        key_rotator, threshold=threshold, delay=delay, rows_per_key=rows_per_key,
        metric_kwargs=metric_kwargs)

    # Remove old incomplete duplicates
    existing_df = existing_df[~existing_df["question"].astype(str).isin(
            new_results_df["question"].astype(str))]

    # Merge
    final_df = pd.concat([existing_df, new_results_df], ignore_index=True)
    if "question_idx" in final_df.columns:
      final_df.drop(columns=["question_idx"], inplace=True)
    final_df = final_df.merge(eval_dataset[["question", "question_idx"]], on="question", how="left")
    cols = ["question_idx"] + [c for c in final_df.columns if c != "question_idx"]
    # final_df = final_df.sort_values("question_idx").reset_index(drop=True)

    final_df.to_csv(existing_results_file, index=False)
    print(f"Completed: {len(final_df)}/{len(eval_dataset)} rows")

    return final_df

In [ ]:
faithfulness_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_rrf_rag_{embedding_key}_faithfulness_{timestamp}.csv"), FaithfulnessMetric,
    de_key_rotator, "Faithfulness", rows_per_key=4, delay=40
)

Need to recompute 6 rows.

6 cases split across 2 key(s) (4 cases/key max):
  Key 0: cases 0–3 (4 cases)
  Key 1: cases 4–5 (2 cases)



Warning: Could not load test run from disk: Expecting value: line 1 column 1 (char 0)

⚠ WARNING: No hyperparameters logged.
» ]8;id=517889;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 8.22s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

⚠ WARNING: No hyperparameters logged.
» ]8;id=839813;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 14.86s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

⚠ WARNING: No hyperparameters logged.
» ]8;id=285365;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 21.36s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 0] Done — 4/4 cases evaluated

=== Faithfulness: 0.9583 (avg over 6 samples) ===
Completed: 200/200 rows


In [12]:
# Defining Answer Correctness
evaluation_steps = [
    "Compare the generated answer with the reference answer in the context of the original biomedical question.",
    "Check whether the generated answer contains factually correct biomedical information and no contradictions to the reference answer.",
    "Verify that all clinically important facts needed to answer the question are present and no critical information is missing.",
    "Ignore wording differences, but penalize incorrect medical claims, unsupported conclusions, or misleading clinical interpretations."
]
deepeval_ac = evaluate_deepeval_parallel(
    test_cases, GEval, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_rrf_rag_{embedding_key}_answer_correctness_{timestamp}.csv"),
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



⚠ WARNING: No hyperparameters logged.
» ]8;id=896353;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

✓ Evaluation completed 🎉! (time taken: 1.27s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

...

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

⚠ WARNING: No hyperparameters logged.
» ]8;id=678335;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.23s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 5] Done — 20/20 cases evaluated

=== AnswerCorrectness [GEval]: 0.6940 (avg over 199 samples) ===
Saved to /content/results/deepeval/hybrid_rrf_rag_minilm_answer_correctness_20260510_081037.csv


In [19]:
answer_correctness = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_rrf_rag_{embedding_key}_answer_correctness_{timestamp}.csv"), GEval,
    de_key_rotator, "AnswerCorrectness [GEval]", rows_per_key=4, delay=40,
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)

Need to recompute 1 rows.

1 cases split across 1 key(s) (4 cases/key max):
  Key 0: cases 0–0 (1 cases)



⚠ WARNING: No hyperparameters logged.
» ]8;id=724195;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.24s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 0] Done — 1/1 cases evaluated

=== AnswerCorrectness [GEval]: 1.0000 (avg over 1 samples) ===
Completed: 200/200 rows


,question,generated_answer,retrieved_contexts,golden_answer,AnswerCorrectness [GEval],question_idx
0,Is there a relationship between rheumatoid art...,"Yes, there is evidence to suggest a relationsh...",['CONCLUSIONS: Based on data derived from self...,Based on data derived from self-reported healt...,1.0,0
1,"Do the changes in the serum levels of IL-2, IL...","Yes, the changes in the serum levels of IL-2, ...",['RESULTS: Seven of the 45 patients (15.5%) de...,The enhancement of serum TNFalpha and IL-6 lev...,0.7,1
2,Does hypoglycaemia increase the risk of cardio...,"Yes, severe hypoglycaemia is associated with a...",['mortality; (iii) CV mortality; and (iv) arrh...,Severe hypoglycaemia is associated with an inc...,0.8,2
3,Telemedicine and type 1 diabetes: is technolog...,"Based on the provided context, the answer is n...","['AIM: In the TELEDIAB-1 study, the Diabeo sys...",The Diabeo system improved glycaemic control i...,0.3,3
4,Can elevated troponin I levels predict complic...,"Yes, elevated troponin I levels can predict co...",['OBJECTIVE: The purpose of this study was to ...,Our results indicate that elevated cTnI levels...,0.6,4
...,...,...,...,...,...,...
195,Do prenatal cigarette exposure and childhood v...,"Yes, according to the provided context, prenat...",['control for early adolescent violence exposu...,prenatal cigarette exposure and childhood viol...,0.8,196
196,Do surgical risk factors and the use of nasoga...,"Yes, surgical risk factors and the use of naso...","[""INTRODUCTION: Although its excellent results...",Surgical risk factors and the use of nasogastr...,1.0,197
197,Does changing resected stomach volume during p...,"According to the provided context, changing re...",['BACKGROUND: Laparoscopic sleeve gastrectomy ...,Changing resected stomach volume during primar...,0.2,198
198,Do DKK3 expression and body mass index differ ...,"Yes, DKK3 expression and body mass index diffe...",['BACKGROUND: Dickkopf-3 (DKK3) may act as a t...,DKK3 expression and body mass index differ in ...,1.0,199


## Exp 2 - Hybrid Retrieval with Cross Encoder Re-Ranking

In [ ]:
CROSS_ENCODER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_cross_encoder(query, docs, top_k=None, reranker=None):
    """Method to rerank retrieved chunks using a cross-encoder.

    Computes semantic relevance scores for each query-document pair
    using a cross-encoder model, then returns the highest ranked
    chunks for final answer generation.

    Args:
        query: User query string used for reranking.
        docs: List of LangChain Document objects retrieved from
            the hybrid retrieval pipeline.
        top_k: Final number of top ranked chunks to return.
        reranker: SentenceTransformers CrossEncoder model.

    Returns:
        List of reranked LangChain Document objects.
    """
    top_k = top_k or config.TOP_K
    reranker = reranker or CROSS_ENCODER
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)
    ranked_docs = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked_docs[:top_k]]


def invoke_hybrid_retriever(query, dense_retriever, sparse_retriever, top_k=None):
    """Method to perform hybrid retrieval using dense and sparse retrievers.

    Retrieves candidate chunks independently using dense semantic retrieval
    (cosine similarity) and sparse lexical retrieval (BM25), removes
    duplicates using chunk metadata, then reranks using a cross-encoder.

    Args:
        query: User query string to retrieve relevant chunks for.
        dense_retriever: Dense retriever object based on vector similarity.
        sparse_retriever: Sparse retriever object based on BM25 scoring.
        top_k: Final number of top ranked chunks to return.

    Returns:
        List of reranked LangChain Document objects.
    """
    top_k = top_k or config.TOP_K
    dense_docs = dense_retriever.invoke(query)
    sparse_docs = sparse_retriever.invoke(query)

    unique_docs = {}
    for doc in dense_docs + sparse_docs:
        key = (doc.metadata["pubid"], doc.metadata["chunk_index"])
        if key not in unique_docs:
          unique_docs[key] = doc

    return rerank_cross_encoder(query, list(unique_docs.values()), top_k=top_k)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Run Hybrid Retrieval RAG with Cross Encoder

In [ ]:
embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
key_rotator = GroqKeyRotator()
llm = key_rotator.get_llm()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Embedding: {config.EMBEDDING_MODELS[embedding_key]}")
print(f"LLM: {key_rotator.model}")
print(f"Run timestamp: {timestamp}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized GroqKeyRotator with 10 API key(s)
Embedding: sentence-transformers/all-MiniLM-L6-v2
LLM: llama-3.3-70b-versatile
Run timestamp: 20260510_062636


In [ ]:
eval_dataset = run_rag_parallel((cosine_retriever, bm25_retriever), golden_df, key_rotator)
eval_dataset.to_csv(str(config.RESULTS_EVALSETS_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_chroma_cosine_{timestamp}.csv"))
print(f"Generated {len(eval_dataset)} answers")


200 rows split across 10 key(s) (20 rows/key max):
  Key 0: rows 0–19 (20 rows)
  Key 1: rows 20–39 (20 rows)
  Key 2: rows 40–59 (20 rows)
  Key 3: rows 60–79 (20 rows)
  Key 4: rows 80–99 (20 rows)
  Key 5: rows 100–119 (20 rows)
  Key 6: rows 120–139 (20 rows)
  Key 7: rows 140–159 (20 rows)
  Key 8: rows 160–179 (20 rows)
  Key 9: rows 180–199 (20 rows)

[Key 4] Done — 20/20 rows collected
[Key 1] Done — 20/20 rows collected
[Key 3] Done — 20/20 rows collected
[Key 7] Done — 20/20 rows collected
[Key 9] Done — 20/20 rows collected
[Key 2] Done — 20/20 rows collected
[Key 6] Done — 20/20 rows collected
[Key 5] Done — 20/20 rows collected
[Key 8] Done — 20/20 rows collected
[Key 0] Done — 20/20 rows collected

Completed 200/200 questions total

Average Time Per Query : 7.563215546384986

Average Total Tokens Per Query : 1804.615
Generated 200 answers


### RAGAS Evaluation

In [ ]:
ragas_cr_scores, ragas_cr_avg, ragas_cr_df = evaluate_ragas(
    eval_dataset, NonLLMContextRecall(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_context_recall_{timestamp}.csv")
)


=== NonLLMContextRecall: 0.2071 (avg over 200 samples) ===
Saved scores to /content/results/ragas/hybrid_cross_encoder_rag_minilm_context_recall_20260510_062636.csv


In [ ]:
ragas_cp_scores, ragas_cp_avg, ragas_cp_df = evaluate_ragas(
    eval_dataset, NonLLMContextPrecisionWithReference(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_context_precision_{timestamp}.csv")
)


=== NonLLMContextPrecisionWithReference: 0.3050 (avg over 200 samples) ===
Saved scores to /content/results/ragas/hybrid_cross_encoder_rag_minilm_context_precision_20260510_062636.csv


In [ ]:
ragas_blue_scores, ragas_blue_avg, ragas_blue_df = evaluate_ragas(
    eval_dataset, BleuScore(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_bleu_{timestamp}.csv")
)


=== BleuScore: 0.1830 (avg over 200 samples) ===
Saved scores to /content/results/ragas/hybrid_cross_encoder_rag_minilm_bleu_20260510_062636.csv


In [ ]:
ragas_rouge_scores, ragas_rouge_avg, ragas_rouge_df = evaluate_ragas(
    eval_dataset, RougeScore(rouge_type="rougeL", mode="fmeasure"),
    results_file=str(config.RESULTS_RAGAS_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_rouge_{timestamp}.csv")
)


=== RougeScore: 0.3007 (avg over 200 samples) ===
Saved scores to /content/results/ragas/hybrid_cross_encoder_rag_minilm_rouge_20260510_062636.csv


In [ ]:
combined_ragas = build_ragas_combined(
    eval_dataset, [ragas_cr_df, ragas_cp_df, ragas_blue_df, ragas_rouge_df], results_file=str(config.RESULTS_RAGAS_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_combined_{timestamp}.csv"))

Saved combined RAGAS results to /content/results/ragas/hybrid_cross_encoder_rag_minilm_combined_20260510_062636.csv


### DeepEval Evaluation

In [20]:
timestamp = "20260510_062636"
eval_dataset = pd.read_csv(str(config.RESULTS_EVALSETS_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_chroma_cosine_{timestamp}.csv"))
eval_dataset['golden_contexts'] = eval_dataset['golden_contexts'].apply(literal_eval)
eval_dataset['retrieved_contexts'] = eval_dataset['retrieved_contexts'].apply(literal_eval)
eval_dataset.head(2)

,Unnamed: 0,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,retrieved_contexts,generated_answer,total_time,prompt_tokens,completion_tokens,total_tokens
0,0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841'],[AIM: The aim of this study was to determine w...,"Yes, there is evidence to suggest a relationsh...",13.765426,1626,125,1751
1,1,1,"Do the changes in the serum levels of IL-2, IL...",The enhancement of serum TNFalpha and IL-6 lev...,[Acute pancreatitis is the major complication ...,Single-hop,['18670651'],[RESULTS: Seven of the 45 patients (15.5%) dev...,"Yes, the changes in the serum levels of IL-2, ...",11.263750,1831,127,1958


In [21]:
test_cases = build_test_cases(eval_dataset)

de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")

Initialized GroqKeyRotator with 10 API key(s)


In [ ]:
deepeval_cr, deepeval_cr_df = evaluate_deepeval_parallel(
    test_cases, ContextualRecallMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_ctx_recall_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



[Key 7] Done — 20/20 cases evaluated

=== Contextual Recall: 0.9108 (avg over 200 samples) ===
Saved to /content/results/deepeval/hybrid_cross_encoder_rag_minilm_ctx_recall_20260510_062636.csv


In [ ]:
deepeval_cp, deepeval_cp_df = evaluate_deepeval_parallel(
    test_cases, ContextualPrecisionMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_ctx_precision_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



[Key 7] Done — 20/20 cases evaluated

=== Contextual Precision: 0.9404 (avg over 200 samples) ===
Saved to /content/results/deepeval/hybrid_cross_encoder_rag_minilm_ctx_precision_20260510_062636.csv


In [ ]:
deepeval_f, deepeval_f_df = evaluate_deepeval_parallel(
    test_cases, FaithfulnessMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_faithfulness_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



⚠ WARNING: No hyperparameters logged.
» ]8;id=957258;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.64s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

✓ Evaluation completed 🎉! (time taken: 21.42s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 2] Done — 20/20 cases evaluated
[Key 8] Error on case 18/20: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kr6pjspgfk2bb9d148pkbzce` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199790, Requested 1108. Please try again in 6m27.936s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
[Key 9] Error on case 19/20: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kr6pmr2ff3jbhg3emgs9qds4` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199336, Requested 1369. Please try again in 5m4.56s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
[Key 6] Error on case 20/20: Error code: 429 - {'error': {'message': 'Rate limit reached for mo

⚠ WARNING: No hyperparameters logged.
» ]8;id=26945;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 63.84s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 3] Done — 20/20 cases evaluated

=== Faithfulness: 0.9815 (avg over 184 samples) ===
Saved to /content/results/deepeval/hybrid_cross_encoder_rag_minilm_faithfulness_20260510_062636.csv


In [ ]:
faithfulness_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_faithfulness_{timestamp}.csv"), FaithfulnessMetric,
    de_key_rotator, "Faithfulness", rows_per_key=4, delay=40
)

Need to recompute 16 rows.

16 cases split across 4 key(s) (4 cases/key max):
  Key 0: cases 0–3 (4 cases)
  Key 1: cases 4–7 (4 cases)
  Key 2: cases 8–11 (4 cases)
  Key 3: cases 12–15 (4 cases)



⚠ WARNING: No hyperparameters logged.
» ]8;id=432427;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.76s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

✓ Evaluation completed 🎉! (time taken: 13.38s | token cost: None)
» Test Results (4 total tests):
   » Pass Rate: 75.0% | Passed: 3 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 0] Done — 4/4 cases evaluated
[Key 3] Error on case 4/4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kr8fn0ftemntmx2tkze0m7p1` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6829, Requested 2594. Please try again in 10.6725s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
[Key 3] Done — 3/4 cases evaluated


⚠ WARNING: No hyperparameters logged.
» ]8;id=449030;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 28.07s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 2] Done — 1/4 cases evaluated

=== Faithfulness: 0.9833 (avg over 12 samples) ===
Completed: 196/200 rows


In [ ]:
faithfulness_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_faithfulness_{timestamp}.csv"), FaithfulnessMetric,
    de_key_rotator, "Faithfulness", rows_per_key=4, delay=30
)

Need to recompute 4 rows.

4 cases split across 1 key(s) (4 cases/key max):
  Key 0: cases 0–3 (4 cases)



⚠ WARNING: No hyperparameters logged.
» ]8;id=692724;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.75s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

⚠ WARNING: No hyperparameters logged.
» ]8;id=766879;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 9.59s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

⚠ WARNING: No hyperparameters logged.
» ]8;id=792227;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 14.61s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

⚠ WARNING: No hyperparameters logged.
» ]8;id=174085;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 16.86s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 0] Done — 4/4 cases evaluated

=== Faithfulness: 1.0000 (avg over 4 samples) ===
Completed: 200/200 rows


In [22]:
deepeval_ac = evaluate_deepeval_parallel(
    test_cases, GEval, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_answer_correctness_{timestamp}.csv"),
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



⚠ WARNING: No hyperparameters logged.
» ]8;id=985318;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.23s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

⚠ WARNING: No hyperparameters logged.
» ]8;id=127519;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.53s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 7] Done — 20/20 cases evaluated

=== AnswerCorrectness [GEval]: 0.7355 (avg over 200 samples) ===
Saved to /content/results/deepeval/hybrid_cross_encoder_rag_minilm_answer_correctness_20260510_062636.csv


In [ ]:
deepeval_ar, deepeval_ar_df = evaluate_deepeval_parallel(
    test_cases, AnswerRelevancyMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"hybrid_cross_encoder_rag_{embedding_key}_ans_relevancy_{timestamp}.csv")
)